In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
%fs
ls /databricks-datasets/nyctaxi/tables/nyctaxi_yellow

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/_delta_log/,_delta_log/,0,1758891154059
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,374549044,1605327908000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,189069652,1605327913000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,373889711,1605327913000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,376396848,1605327908000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,364876497,1605327908000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,363016752,1605327947000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,203737885,1605327989000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,365327633,1605327990000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,355258373,1605328000000


In [0]:
%sql
describe formatted delta.`dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow`

col_name,data_type,comment
vendor_id,string,null
pickup_datetime,timestamp,null
dropoff_datetime,timestamp,null
passenger_count,int,null
trip_distance,double,null
pickup_longitude,double,null
pickup_latitude,double,null
rate_code_id,int,null
store_and_fwd_flag,string,null
dropoff_longitude,double,null


In [0]:
%sql
DROP TABLE IF EXISTS demo_taxi_200files;
CREATE TABLE demo_taxi_200files (
  vendor_id STRING,
  pickup_datetime TIMESTAMP,
  dropoff_datetime TIMESTAMP,
  passenger_count INT,
  trip_distance DOUBLE,
  pickup_longitude DOUBLE,
  pickup_latitude DOUBLE,
  rate_code_id INT,
  store_and_fwd_flag STRING,
  dropoff_longitude DOUBLE,
  dropoff_latitude DOUBLE,
  payment_type STRING,
  fare_amount DOUBLE,
  extra DOUBLE,
  mta_tax DOUBLE,
  tip_amount DOUBLE,
  tolls_amount DOUBLE,
  total_amount DOUBLE
)
USING DELTA
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
);


In [0]:
table_location = "dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

# List data files (ignore _delta_log)
all_files = [f.path for f in dbutils.fs.ls(table_location) if f.path.endswith(".parquet")]

# Pick first 10 files
files10 = all_files[:10]
print("Using these 10 files:", files10)

df_10 = spark.read.parquet(*files10)

df_10.repartition(200).write.format('delta').mode("overwrite").saveAsTable("demo_taxi_200files")


Using these 10 files: ['dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b

In [0]:
%sql
select * from demo_taxi_200files;

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
CMT,2009-01-01T00:11:17.000Z,2009-01-01T00:18:06.000Z,1,1.3,-73.996867,40.747527,null,null,-74.002836,40.760508,Cash,7.0,0.0,null,0.0,0.0,7.0
CMT,2009-01-01T00:46:25.000Z,2009-01-01T00:56:35.000Z,1,1.4,-73.990467,40.740552,null,null,-73.994455,40.748689,Cash,7.8,0.0,null,0.0,0.0,7.8
CMT,2009-01-01T01:36:24.000Z,2009-01-01T01:44:11.000Z,1,2.0,-73.996135,40.738305,null,null,-73.977755,40.764248,Cash,12.2,0.0,null,0.0,0.0,12.2
CMT,2009-01-01T02:36:39.000Z,2009-01-01T03:05:50.000Z,3,14.3,-73.990879,40.749269,null,null,-73.898183,40.905664,Cash,34.2,0.0,null,0.0,0.0,34.2
CMT,2009-01-01T03:32:11.000Z,2009-01-01T03:39:03.000Z,3,1.7,-73.988816,40.748764,null,null,-73.987902,40.728732,Cash,7.0,0.0,null,0.0,0.0,7.0
CMT,2009-01-01T04:37:56.000Z,2009-01-01T04:42:11.000Z,1,0.5,-73.990111,40.737,null,null,-73.985633,40.73263,Cash,5.0,0.0,null,0.0,0.0,5.0
CMT,2009-01-01T07:10:32.000Z,2009-01-01T07:18:32.000Z,1,2.0,-73.991592,40.750106,null,null,-73.968832,40.75043,Cash,7.3,0.0,null,0.0,0.0,7.3
CMT,2009-01-01T11:07:13.000Z,2009-01-01T11:14:40.000Z,1,1.3,-73.990427,40.737196,null,null,-73.982685,40.742672,Credit,6.5,0.0,null,1.0,0.0,7.5
CMT,2009-01-01T13:14:49.000Z,2009-01-01T13:19:18.000Z,2,0.6,-73.993355,40.749733,null,null,-73.988753,40.757225,Cash,4.5,0.0,null,0.0,0.0,4.5
CMT,2009-01-01T14:46:03.000Z,2009-01-01T14:55:54.000Z,2,2.8,-73.995873,40.743511,null,null,-74.011469,40.707896,Cash,8.9,0.0,null,0.0,0.0,8.9


In [0]:
%sql
select count(*) from demo_taxi_200files;

count(*)
90848212


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance);

min(trip_distance),max(trip_distance),file_name
0.0,53.8,part-00065-6ba92534-661f-4040-ae77-e2356d6e0369.c000.snappy.parquet
0.0,89.0,part-00177-8e0a15d5-3cac-4320-a95e-19363fe2a665.c000.snappy.parquet
0.0,504.1,part-00064-cf672275-ac5e-4783-b81a-65eac8991e55.c000.snappy.parquet
0.0,50000.0,part-00108-9398876f-4d49-44e0-b4f8-64593c2edf73.c000.snappy.parquet
0.0,175.6,part-00133-43caf005-9c0a-4797-97aa-1c22cdfdf0b3.c000.snappy.parquet
0.0,183.1,part-00146-85e4f84c-c646-47ed-aaf1-8f7985158f9c.c000.snappy.parquet
0.0,174.4,part-00165-cf341c5e-02ac-411f-8f86-545a91809564.c000.snappy.parquet
0.0,364.1,part-00093-c30bbf73-c29c-4286-b10b-67f4d0f9d6ba.c000.snappy.parquet
0.0,96.7,part-00048-4e204bd3-5fe5-415e-8b0d-87e59e645878.c000.snappy.parquet
0.0,81.0,part-00011-61eda58b-42b5-4feb-b682-9386743c67e1.c000.snappy.parquet


In [0]:
%sql
optimize demo_taxi_200files zorder by (trip_distance);
    


path,metrics
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/227c8766-af2e-4f3e-9637-513995309870,"List(45, 200, List(52352997, 92182448, 6.4768034822222225E7, 45, 2914561567), List(15184503, 15260454, 1.5232821105E7, 200, 3046564221), 0, List(minCubeSize(107374182400), List(0, 0), List(200, 3046564221), 0, List(200, 3046564221), 1, null), null, 0, 1, 200, 0, false, 0, 0, 1758891937386, 1758891979488, 16, 1, null, List(0, 0), null, 18, 18, 271491, 0, null)"


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance);

min(trip_distance),max(trip_distance),file_name
0.0,0.3,part-00000-33482e25-f473-4518-a639-82d1ebc0b86d.c000.snappy.parquet
0.3,0.5,part-00001-85424b0f-1ff3-48d8-9a86-e58400bf0c0a.c000.snappy.parquet
0.5,0.59,part-00002-c5600629-92b1-4d34-bba8-ee80ba628b9c.c000.snappy.parquet
0.59,0.64,part-00003-8c69ed57-9b17-4cca-b639-5ead0e657d8a.c000.snappy.parquet
0.64,0.7,part-00004-26712a14-df52-4b25-982f-f6874a7415e4.c000.snappy.parquet
0.7,0.8,part-00005-f86e5c75-cf39-413e-af08-080d09513228.c000.snappy.parquet
0.8,0.84,part-00006-09de54aa-9467-47dc-b9ab-1560ce96e31a.c000.snappy.parquet
0.84,0.9,part-00007-97ae196a-e4ba-4b29-a801-1c493b8f26d6.c000.snappy.parquet
0.9,0.97,part-00008-2cf3e2c0-f14b-46ba-9e8a-08c793ef70cf.c000.snappy.parquet
0.97,1.0,part-00009-2a0a183f-14bf-4b39-a0aa-5040c7a201e8.c000.snappy.parquet


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 5 and trip_distance < 7;

count(*)
6065575
